# CS781 Phase 1 - LegalLM v2: continue training + stronger decoding

Builds on your first LegalLM run (109.5M params, 1.16B tokens, leaderboard BERTScore F1 **0.8427**). Attach that run's output as input; this notebook does not start from zero.

### What the first run showed, and what this version changes

| Finding from run 1 | Change in v2 |
|---|---|
| Local score 0.8593 overall, but **0.8450 on Supreme Court prompts**, which is close to the leaderboard. The Acts prompts (easy, formulaic) inflated the local number, and the test prompts are court judgments. | Decoding is chosen on **judgment prompts only** (Supreme Court, High Courts, Tribunals), so the local score tracks the leaderboard. |
| Validation loss still falling when training stopped (1.16B tokens; ~2.2B is compute-optimal for 110M params) | **~8 h more training** from the trained weights (warm restart, new LR decay) on a judgment-heavy mix with **fresh High Court documents** |
| MBR (8 samples) gave +0.007 over the best plain decoding | **Bigger MBR:** both GPUs generate (about 2x faster), each candidate is BERTScore-encoded once instead of once per pair (exact same scores, checked at runtime), so many more candidates fit in the time |
| Repetition penalty 1.2 > 1.1 > none; n-gram blocking hurt | Sweep around 1.2 (1.15 - 1.4); n-gram blocking dropped |
| 2 of 1000 outputs were shorter than 128 words | Longer generation budget; MBR never picks a short candidate when a full-length one exists |
| - | Old and new model scored on the **same** prompts; the better one is used. A plain-decoding `submission.zip` is written first as a fallback, then replaced by the MBR one |

### How to run
1. **Add Input -> Your Work -> Notebooks ->** the run that produced the 0.8427 submission (it contains `cs781_scratch/legal-lm-final` and `cs781_scratch/checkpoint/ckpt.pt`).
2. **GPU T4 x2**, Internet on, secret `capstoned`.
3. **Save Version -> Save & Run All**. About 11.5 h. For a quick decoding-only run (~2.5 h, no training), set `TRAIN_MORE = False`.

## 1. Hugging Face Login

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("capstoned")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("Hugging Face login successful.")

## 2. GPUs and Packages

In [ ]:
import sys, subprocess
import torch

N_GPUS = torch.cuda.device_count()
print("CUDA available:", torch.cuda.is_available(), "| GPUs:", N_GPUS)
for i in range(N_GPUS):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
for pkg in ["bert_score", "rouge_score", "sacrebleu", "tokenizers", "ijson", "psutil"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("Packages ready.")

## 3. Configuration

In [ ]:
import os, sys, json, time, math, random, re, zipfile, shutil, contextlib, subprocess
import numpy as np

NOTEBOOK_START = time.time()

# ---- What to do
TRAIN_MORE = True             # False = decoding-only run (~2.5 h) with the first-run model

# ---- Data
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_PREFIX = "Phase_1_Pretraining/Train/"
SUPREME_COURT_FILE = "Phase_1_Pretraining/Train/train_supreme_court.json"
TEST_FILE = "Phase_1_Pretraining/Test/test.json"
SEED = 42
SC_VAL_FRACTION = 0.05        # identical held-out split to both earlier runs (never trained on)
OTHER_VAL_FRACTION = 0.01
VAL_TOKENS = 3_000_000
SAMPLE_SEED = 7               # different from run 1 -> mostly new High Court / Tribunal documents
# Continued-training mix (token caps per file; ~8 h of training uses ~1.0B tokens)
FILE_TOKEN_CAPS = {"acts": 60_000_000, "high_court": 800_000_000,
                   "supreme_court": 300_000_000, "tribunal": 300_000_000}

def file_kind(fname):
    n = fname.lower()
    for k in ["acts", "high_court", "supreme_court", "tribunal"]:
        if k in n:
            return k
    return "other"

# ---- Continued training (warm restart from the trained model)
MIN_PARAMS, MAX_PARAMS = 50_000_000, 300_000_000
MICRO_BATCH = 8
GRAD_ACCUM = 16               # 8 x 16 x 2 GPUs x 1024 = 262k tokens per step (same as run 1)
PEAK_LR = 3e-4                # half of run 1's peak: the model is already trained
MIN_LR_RATIO = 0.1
WARMUP_STEPS = 300
DECAY_FRACTION = 0.3
WEIGHT_DECAY = 0.1
USE_COMPILE = True

# ---- Time plan (12 h Kaggle limit)
SESSION_LIMIT_HOURS = 11.5
POST_TRAIN_RESERVE_HOURS = 2.4
MAX_TRAIN_HOURS = 8.5
MIN_TRAIN_HOURS = 0.5

# ---- Evaluation (judgment prompts only; the test set is court judgments)
TARGET_WORDS = 128
EVAL_PAIRS_PER_KIND = {"supreme_court": 90, "high_court": 90, "tribunal": 40}
SWEEP_EXAMPLES = 110          # prompts used to choose decoding
REP_PENALTIES = [1.15, 1.2, 1.25, 1.3, 1.4]
MBR_POOL_TEMPS = [0.5, 0.75]  # sampling temperatures tried for MBR candidates
MBR_POOL_SAMPLES = 24         # samples per temperature in the decoding search
MBR_MAX_SAMPLES = 48          # cap for the test set (actual number is chosen to fit the time left)
MBR_MIN_SAMPLES = 8
SEQ_BATCH = 96                # sequences per generation batch per GPU
FINAL_MARGIN_HOURS = 0.3

# ---- Paths
OUTPUT_DIR = "/kaggle/working/cs781_v2"
CODE_DIR = os.path.join(OUTPUT_DIR, "code")            # must match the %%writefile paths below
CKPT_DIR = os.path.join(OUTPUT_DIR, "checkpoint")
CKPT_PATH = os.path.join(CKPT_DIR, "ckpt.pt")
TRAIN_LOG_PATH = os.path.join(CKPT_DIR, "train_log.json")
FINAL_MODEL_DIR = os.path.join(OUTPUT_DIR, "legal-lm-final")
CONT_MODEL_DIR = os.path.join(OUTPUT_DIR, "legal-lm-continued")
TOKENIZER_PATH = os.path.join(OUTPUT_DIR, "tokenizer.json")
DATA_DIR = "/tmp/cs781_data"
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, "training_validation_loss.png")
METRICS_PATH = os.path.join(OUTPUT_DIR, "evaluation_metrics.json")
CANDIDATES_PATH = os.path.join(OUTPUT_DIR, "test_candidates.json")
for d in [OUTPUT_DIR, CODE_DIR, CKPT_DIR, DATA_DIR]:
    os.makedirs(d, exist_ok=True)

def hours_left():
    return SESSION_LIMIT_HOURS - (time.time() - NOTEBOOK_START) / 3600

random.seed(SEED)
np.random.seed(SEED)
print("Configuration loaded.")

## 4. Find the First Run (attached input)

#### Needs its `tokenizer.json` (the vocabulary must not change) and its model. `ckpt.pt` also carries the optimizer state, which makes the warm restart smoother; the plain model file works too.

In [ ]:
def find_previous_run():
    hits = []
    for root in ["/kaggle/input", "/kaggle/working"]:
        if not os.path.isdir(root):
            continue
        for dirpath, _, filenames in os.walk(root):
            if dirpath.startswith(OUTPUT_DIR):
                continue
            if os.path.basename(dirpath) == "legal-lm-final" and "config.json" in filenames \
                    and "pytorch_model.bin" in filenames:
                hits.append(dirpath)
    if not hits:
        raise FileNotFoundError(
            "First LegalLM run not found. Add Input -> Your Work -> Notebooks -> the notebook that made the "
            "0.8427 submission (the version that finished), then run again.")
    return sorted(hits)[0]

PREV_MODEL_DIR = find_previous_run()
PREV_RUN_DIR = os.path.dirname(PREV_MODEL_DIR)
prev_tok = os.path.join(PREV_MODEL_DIR, "tokenizer.json")
if not os.path.exists(prev_tok):
    prev_tok = os.path.join(PREV_RUN_DIR, "tokenizer.json")
shutil.copy(prev_tok, TOKENIZER_PATH)
PREV_CKPT = os.path.join(PREV_RUN_DIR, "checkpoint", "ckpt.pt")
INIT_FROM = PREV_CKPT if os.path.exists(PREV_CKPT) else os.path.join(PREV_MODEL_DIR, "pytorch_model.bin")
with open(os.path.join(PREV_MODEL_DIR, "config.json")) as f:
    MODEL = json.load(f)
prev_state = {}
if os.path.exists(os.path.join(PREV_MODEL_DIR, "training_state.json")):
    prev_state = json.load(open(os.path.join(PREV_MODEL_DIR, "training_state.json")))
print("First run model :", PREV_MODEL_DIR)
print("Warm start from :", INIT_FROM)
print("Architecture    :", MODEL)
print("Previously trained on:", f"{prev_state.get('tokens', 0):,} tokens")

## 5. Tokenizer (from the first run) and Test Data

In [ ]:
from tokenizers import Tokenizer
from huggingface_hub import hf_hub_download, HfApi
import ijson, psutil

def ram():
    vm = psutil.virtual_memory()
    return f"RAM {vm.used / 1e9:.1f}/{vm.total / 1e9:.0f} GB"

tok = Tokenizer.from_file(TOKENIZER_PATH)
EOS_ID = tok.token_to_id("<|endoftext|>")
assert tok.get_vocab_size() == MODEL["vocab_size"], "tokenizer and model vocabulary differ"
ENCODE_BATCH = getattr(tok, "encode_batch_fast", tok.encode_batch)

test_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=TEST_FILE, repo_type="dataset", token=HF_TOKEN)
with open(test_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)
test_word_counts = np.array([len(r["text"].split()) for r in test_data.values()])
TEST_PROMPT_WORDS = int(np.median(test_word_counts))
test_tok = [len(e.ids) for e in ENCODE_BATCH([r["text"] for r in test_data.values()])]
print(f"Test examples: {len(test_data)} | prompt words: median {TEST_PROMPT_WORDS}, max {test_word_counts.max()} "
      f"| prompt tokens: median {np.median(test_tok):.0f}, max {max(test_tok)}")
MAX_NEW_TOKENS = int(min(MODEL["max_seq_len"] - np.percentile(test_tok, 95), 360))
print("MAX_NEW_TOKENS:", MAX_NEW_TOKENS)

## 6. Stream the Corpus: Training Mix, Validation Tokens and Evaluation Documents

#### Held-out documents are the same as in run 1 (same seed-42 draw per document), so they were never trained on. Training documents are re-sampled with a new seed, so most High Court and Tribunal documents are new to the model.

In [ ]:
listing = HfApi().list_repo_files(HF_DATASET_REPO, repo_type="dataset", token=HF_TOKEN)
TRAIN_FILES = sorted(f for f in listing if f.startswith(TRAIN_PREFIX) and f.endswith(".json"))
LOCAL = {f: hf_hub_download(repo_id=HF_DATASET_REPO, filename=f, repo_type="dataset", token=HF_TOKEN)
         for f in TRAIN_FILES}
FILE_BYTES = {f: os.path.getsize(p) for f, p in LOCAL.items()}
for f in TRAIN_FILES:
    print(f"  {f}: {FILE_BYTES[f] / 1e9:.2f} GB | kind: {file_kind(f)}")

def iter_docs(fname):
    # Yields (is_validation, text) per document in file order, streaming. Same draw as runs 1 and 2.
    rng = random.Random(SEED)
    frac = SC_VAL_FRACTION if fname == SUPREME_COURT_FILE else OTHER_VAL_FRACTION
    path = LOCAL[fname]
    with open(path, "rb") as fh:
        head = fh.read(4096).lstrip()
    with open(path, "rb") as fh:
        items = (ijson.items(fh, "item", use_float=True) if head[:1] == b"["
                 else (v for _, v in ijson.kvitems(fh, "", use_float=True)))
        for rec in items:
            text = rec.get("text") if isinstance(rec, dict) else None
            yield rng.random() < frac, (text if isinstance(text, str) else "")

TRAIN_BIN = os.path.join(DATA_DIR, "train.bin")
VAL_BIN = os.path.join(DATA_DIR, "val.bin")
BYTES_PER_TOKEN_EST = 4.5
MIN_EVAL_DOC_WORDS = TEST_PROMPT_WORDS + TARGET_WORDS + 20

def write_tokens(fh, texts):
    if not texts:
        return 0
    arr = np.concatenate([np.asarray(e.ids + [EOS_ID], dtype=np.uint16) for e in ENCODE_BATCH(texts)])
    arr.tofile(fh)
    return len(arr)

t0 = time.time()
eval_docs = {}
val_per_file = VAL_TOKENS // len(TRAIN_FILES)
total_train = total_val = 0
with open(TRAIN_BIN, "wb") as ftr, open(VAL_BIN, "wb") as fva:
    for fname in TRAIN_FILES:
        kind = file_kind(fname)
        cap = FILE_TOKEN_CAPS.get(kind, 100_000_000) if TRAIN_MORE else 0
        keep_p = min(1.0, 1.3 * cap * BYTES_PER_TOKEN_EST / max(1, FILE_BYTES[fname]))
        need_eval = EVAL_PAIRS_PER_KIND.get(kind, 0)
        res_size = 3 * need_eval
        rng_keep, rng_res = random.Random(SAMPLE_SEED), random.Random(SEED + 2)
        n_docs = n_tr = n_va = n_long = 0
        tr_buf, va_buf, reservoir = [], [], []
        for is_val, text in iter_docs(fname):
            n_docs += 1
            if not text:
                continue
            if is_val:
                if n_va < val_per_file:
                    va_buf.append(text)
                    if len(va_buf) >= 64:
                        n_va += write_tokens(fva, va_buf); va_buf = []
                if res_size and len(text.split()) >= MIN_EVAL_DOC_WORDS:
                    n_long += 1
                    if len(reservoir) < res_size:
                        reservoir.append(text)
                    else:
                        j = rng_res.randrange(n_long)
                        if j < res_size:
                            reservoir[j] = text
            elif n_tr < cap and rng_keep.random() <= keep_p:
                tr_buf.append(text)
                if len(tr_buf) >= 256:
                    n_tr += write_tokens(ftr, tr_buf); tr_buf = []
            elif not TRAIN_MORE and n_va >= val_per_file and res_size == 0:
                break
        n_tr += write_tokens(ftr, tr_buf)
        if n_va < val_per_file:
            n_va += write_tokens(fva, va_buf)
        eval_docs[fname] = reservoir
        total_train += n_tr
        total_val += n_va
        print(f"  {fname}: {n_docs:,} docs | train {n_tr / 1e6:,.1f}M tokens | val {n_va / 1e6:.2f}M | "
              f"eval docs {len(reservoir)} | {(time.time() - t0) / 60:.1f} min | {ram()}", flush=True)
print(f"Train pool: {total_train / 1e6:,.1f}M tokens | validation: {total_val / 1e6:.2f}M | {(time.time() - t0) / 60:.1f} min")

## 7. Evaluation Pairs (judgments only)

#### Each pair: a 512-word prompt (the test-prompt length) cut from a held-out judgment, and the true next 128 words as reference. Interleaved across Supreme Court / High Courts / Tribunals.

In [ ]:
rng = random.Random(SEED)
eval_pairs = []
by_kind = {}
for fname, docs in eval_docs.items():
    kind = file_kind(fname)
    want = EVAL_PAIRS_PER_KIND.get(kind, 0)
    pairs = []
    for text in docs:
        if len(pairs) >= want:
            break
        spans = [m.start() for m in re.finditer(r"\S+", text)]
        words = text.split()
        plen = min(TEST_PROMPT_WORDS, len(words) - TARGET_WORDS)
        c = rng.randint(plen, len(words) - TARGET_WORDS)
        pairs.append({"kind": kind, "prompt": text[spans[c - plen]:spans[c]].rstrip(),
                      "reference": " ".join(words[c:c + TARGET_WORDS])})
    by_kind[kind] = pairs
mx = max(len(v) for v in by_kind.values())
for i in range(mx):
    for kind in sorted(by_kind):
        if i < len(by_kind[kind]):
            eval_pairs.append(by_kind[kind][i])
eval_prompts = [p["prompt"] for p in eval_pairs]
eval_refs = [p["reference"] for p in eval_pairs]
eval_kinds = [p["kind"] for p in eval_pairs]
with open(os.path.join(OUTPUT_DIR, "eval_pairs.json"), "w", encoding="utf-8") as fh:
    json.dump(eval_pairs, fh, ensure_ascii=False)
print("Evaluation pairs:", len(eval_pairs), {k: len(v) for k, v in by_kind.items()})
SWEEP_N = min(SWEEP_EXAMPLES, len(eval_pairs))

## 8. Model Architecture (`model.py`, unchanged from run 1)

In [ ]:
%%writefile /kaggle/working/cs781_v2/code/model.py
"""LegalLM: a decoder-only transformer written from scratch for CS781 Phase 1.

Design (modern "Llama-style" block, all implemented here):
  * RMSNorm (pre-norm) instead of LayerNorm
  * Rotary position embeddings (RoPE) instead of learned absolute positions
  * SwiGLU feed-forward instead of GELU MLP
  * No bias terms, tied input/output embeddings
  * PyTorch scaled_dot_product_attention (memory-efficient kernel on T4)
  * KV cache + left-padding support for fast batched generation
"""
import math
from dataclasses import dataclass, asdict

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class ModelConfig:
    vocab_size: int = 32000
    n_layer: int = 12
    n_head: int = 12
    d_model: int = 768
    d_ff: int = 2048
    max_seq_len: int = 1024
    rope_theta: float = 10000.0
    norm_eps: float = 1e-5

    def to_dict(self):
        return asdict(self)


class RMSNorm(nn.Module):
    def __init__(self, dim, eps):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight).to(x.dtype)


def rope_tables(cfg):
    head_dim = cfg.d_model // cfg.n_head
    inv_freq = 1.0 / (cfg.rope_theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t = torch.arange(cfg.max_seq_len).float()
    freqs = torch.outer(t, inv_freq)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    # x: (B, H, T, head_dim); cos/sin: (B or 1, 1, T, head_dim/2)
    half = x.shape[-1] // 2
    x1, x2 = x[..., :half], x[..., half:]
    cos, sin = cos.to(x.dtype), sin.to(x.dtype)
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)


class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.head_dim = cfg.d_model // cfg.n_head
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=False)
        self.proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)

    def forward(self, x, cos, sin, attn_mask=None, cache=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if cache is not None:
            if "k" in cache:
                k = torch.cat([cache["k"], k], dim=2)
                v = torch.cat([cache["v"], v], dim=2)
            cache["k"], cache["v"] = k, v
        if attn_mask is None:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        else:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)
        return self.proj(y.transpose(1, 2).reshape(B, T, C))


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.w12 = nn.Linear(cfg.d_model, 2 * cfg.d_ff, bias=False)
        self.w3 = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)

    def forward(self, x):
        a, b = self.w12(x).chunk(2, dim=-1)
        return self.w3(F.silu(a) * b)


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = Attention(cfg)
        self.norm2 = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.mlp = SwiGLU(cfg)

    def forward(self, x, cos, sin, attn_mask=None, cache=None):
        x = x + self.attn(self.norm1(x), cos, sin, attn_mask, cache)
        return x + self.mlp(self.norm2(x))


class LegalLM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.tok = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok.weight                      # tied embeddings
        cos, sin = rope_tables(cfg)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init)
        for name, p in self.named_parameters():                 # scaled init of residual projections
            if name.endswith("attn.proj.weight") or name.endswith("mlp.w3.weight"):
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * cfg.n_layer))

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None, positions=None, attn_mask=None, caches=None, last_only=False):
        B, T = idx.shape
        if positions is None:
            cos = self.rope_cos[:T][None, None]
            sin = self.rope_sin[:T][None, None]
        else:
            cos = self.rope_cos[positions].unsqueeze(1)
            sin = self.rope_sin[positions].unsqueeze(1)
        x = self.tok(idx)
        for i, block in enumerate(self.blocks):
            x = block(x, cos, sin, attn_mask, None if caches is None else caches[i])
        x = self.norm(x)
        if targets is not None:
            logits = self.head(x)
            return F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1))
        if last_only:
            x = x[:, -1:]
        return self.head(x)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

In [ ]:
sys.path.insert(0, CODE_DIR)
import importlib
import model as model_module
importlib.reload(model_module)
from model import LegalLM, ModelConfig, count_parameters
_m = LegalLM(ModelConfig(**MODEL))
model_size = count_parameters(_m)
print(f"LegalLM: {model_size / 1e6:.2f}M parameters")
assert MIN_PARAMS <= model_size <= MAX_PARAMS, f"{model_size:,} parameters is outside the allowed range"
del _m

## 9. Training Script (`train.py`)

#### Same as run 1 plus `init_from`: start from the trained weights and optimizer state with a fresh warmup -> constant -> decay schedule.

In [ ]:
%%writefile /kaggle/working/cs781_v2/code/train.py
"""Time-budgeted training for LegalLM. Launched by the notebook with torchrun (one process per GPU).

* DistributedDataParallel across all GPUs (no DataParallel output gathering)
* fp16 autocast + GradScaler (T4 has no bf16), fused AdamW, optional torch.compile
* Random windows from a uint16 token memmap (documents separated by EOS)
* LR: warmup -> constant -> linear decay over the last DECAY_FRACTION of the training time,
  so the run always ends annealed when the time budget runs out
* Periodic checkpoints (resumable across Kaggle sessions)
"""
import json
import os
import sys
import time
from contextlib import nullcontext

import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from model import LegalLM, ModelConfig, count_parameters  # noqa: E402

cfg = json.load(open(sys.argv[1]))
ddp = int(os.environ.get("WORLD_SIZE", "1")) > 1
use_cuda = torch.cuda.is_available()
if ddp:
    dist.init_process_group("nccl" if use_cuda else "gloo")
    rank, local, world = int(os.environ["RANK"]), int(os.environ["LOCAL_RANK"]), int(os.environ["WORLD_SIZE"])
else:
    rank, local, world = 0, 0, 1
if use_cuda:
    torch.cuda.set_device(local)
    device = torch.device("cuda", local)
else:
    device = torch.device("cpu")
master = rank == 0


def log(*a):
    if master:
        print(*a, flush=True)


T = cfg["model"]["max_seq_len"]
B = cfg["micro_batch"]
ACC = cfg["grad_accum"]
amp = torch.autocast("cuda", dtype=torch.float16) if use_cuda else nullcontext()

train_data = np.memmap(cfg["train_bin"], dtype=np.uint16, mode="r")
val_data = np.memmap(cfg["val_bin"], dtype=np.uint16, mode="r")
log(f"Token pool: {len(train_data):,} train / {len(val_data):,} val | GPUs: {world}")

torch.manual_seed(cfg["seed"])
model = LegalLM(ModelConfig(**cfg["model"])).to(device)
log(f"Parameters: {count_parameters(model) / 1e6:.2f}M")

decay, no_decay = [], []
for _, p in model.named_parameters():
    (decay if p.dim() >= 2 else no_decay).append(p)
groups = [{"params": decay, "weight_decay": cfg["weight_decay"]},
          {"params": no_decay, "weight_decay": 0.0}]
opt_kwargs = dict(lr=cfg["peak_lr"], betas=(0.9, 0.95), eps=1e-8)
opt = torch.optim.AdamW(groups, fused=True, **opt_kwargs) if use_cuda else torch.optim.AdamW(groups, **opt_kwargs)
scaler = torch.amp.GradScaler("cuda", enabled=use_cuda)

state = {"step": 0, "train_seconds": 0.0, "tokens": 0, "log": []}
if not os.path.exists(cfg["ckpt_path"]) and cfg.get("init_from"):
    # Warm start: weights (and optimizer moments, if present) from a previous run, fresh LR schedule.
    ck = torch.load(cfg["init_from"], map_location=device, weights_only=False)
    model.load_state_dict(ck["model"] if "model" in ck else ck)
    if "opt" in ck:
        try:
            opt.load_state_dict(ck["opt"])
        except Exception as e:
            log("Optimizer state not reused:", repr(e)[:200])
    prev_tokens = ck.get("state", {}).get("tokens", 0) if isinstance(ck.get("state"), dict) else 0
    del ck
    log(f"Warm start from {cfg['init_from']} (model previously trained on {prev_tokens:,} tokens)")
elif os.path.exists(cfg["ckpt_path"]):
    ck = torch.load(cfg["ckpt_path"], map_location=device, weights_only=False)
    model.load_state_dict(ck["model"])
    opt.load_state_dict(ck["opt"])
    scaler.load_state_dict(ck["scaler"])
    state = ck["state"]
    del ck
    log(f"Resumed at step {state['step']} ({state['tokens']:,} tokens, {state['train_seconds'] / 3600:.2f} h trained)")

rng = np.random.default_rng([cfg["seed"], rank, state["step"]])


def get_batch(r, data=train_data, bsz=B):
    ix = r.integers(0, len(data) - T - 1, size=bsz)
    buf = torch.from_numpy(np.stack([data[i:i + T + 1] for i in ix]).astype(np.int64))
    if use_cuda:
        buf = buf.pin_memory().to(device, non_blocking=True)
    return buf[:, :-1], buf[:, 1:]


# ---- optional torch.compile, tested on one micro-batch before DDP wraps the model
run_model = model
if cfg.get("compile") and use_cuda:
    try:
        t0 = time.time()
        compiled = torch.compile(model)
        x, y = get_batch(np.random.default_rng(0))
        with amp:
            loss = compiled(x, targets=y)
        loss.backward()
        run_model = compiled
        log(f"torch.compile OK ({time.time() - t0:.0f}s)")
    except Exception as e:  # fall back to eager mode
        log("torch.compile failed, using eager mode:", repr(e)[:300])
    model.zero_grad(set_to_none=True)
dmodel = DDP(run_model, device_ids=[local] if use_cuda else None) if ddp else run_model


@torch.no_grad()
def evaluate():
    n_windows = B * world * cfg.get("eval_batches", 4)
    starts = np.linspace(0, len(val_data) - T - 2, n_windows).astype(np.int64)[rank::world]
    total = torch.zeros(2, device=device, dtype=torch.float64)
    run_model.eval()
    for s in range(0, len(starts), B):
        chunk = starts[s:s + B]
        buf = torch.from_numpy(np.stack([val_data[i:i + T + 1] for i in chunk]).astype(np.int64)).to(device)
        with amp:
            loss = run_model(buf[:, :-1], targets=buf[:, 1:])
        total[0] += loss.double() * len(chunk)
        total[1] += len(chunk)
    run_model.train()
    if ddp:
        dist.all_reduce(total)
    return (total[0] / total[1]).item()


def save_ckpt():
    if master:
        tmp = cfg["ckpt_path"] + ".tmp"
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                    "state": state, "config": cfg["model"]}, tmp)
        os.replace(tmp, cfg["ckpt_path"])
        with open(cfg["log_path"], "w") as f:
            json.dump(state["log"], f)
    if ddp:
        dist.barrier()


def lr_at(step, progress):
    warm = min(1.0, (step + 1) / cfg["warmup_steps"])
    start = 1.0 - cfg["decay_fraction"]
    if progress <= start:
        return cfg["peak_lr"] * warm
    frac = min(1.0, (progress - start) / cfg["decay_fraction"])
    return cfg["peak_lr"] * warm * (1.0 - (1.0 - cfg["min_lr_ratio"]) * frac)


loop_start = time.time()
session_seconds = max(0.0, cfg["deadline"] - loop_start)
base_seconds = state["train_seconds"]
total_seconds = cfg.get("total_train_seconds") or (base_seconds + session_seconds)
tokens_per_step = B * T * ACC * world
log(f"Training up to {session_seconds / 3600:.2f} h this session; LR schedule spans "
    f"{total_seconds / 3600:.2f} h in total; {tokens_per_step:,} tokens per step")

last_eval = last_ckpt = t_log = time.time()
tok_log = state["tokens"]
dmodel.train()
while True:
    # rank 0 decides (time, eval, checkpoint) so all processes stay in lock-step
    now = time.time()
    ctrl = torch.tensor([now - loop_start,
                         float(now - last_eval > cfg["eval_every_sec"]),
                         float(now - last_ckpt > cfg["ckpt_every_sec"])], device=device, dtype=torch.float64)
    if ddp:
        dist.broadcast(ctrl, 0)
    elapsed, do_eval, do_ckpt = ctrl[0].item(), ctrl[1].item() > 0, ctrl[2].item() > 0
    trained = base_seconds + elapsed
    if elapsed >= session_seconds or trained >= total_seconds:
        break

    lr = lr_at(state["step"], trained / total_seconds)
    for g in opt.param_groups:
        g["lr"] = lr
    loss_sum = torch.zeros((), device=device)
    for micro in range(ACC):
        x, y = get_batch(rng)
        sync_ctx = dmodel.no_sync() if (ddp and micro < ACC - 1) else nullcontext()
        with sync_ctx:
            with amp:
                loss = dmodel(x, targets=y)
            scaler.scale(loss / ACC).backward()
        loss_sum += loss.detach().float()
    scaler.unscale_(opt)
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(opt)
    scaler.update()
    opt.zero_grad(set_to_none=True)
    state["step"] += 1
    state["tokens"] += tokens_per_step
    state["train_seconds"] = trained

    if state["step"] % cfg["log_every"] == 0:
        mean_loss = loss_sum / ACC
        if ddp:
            dist.all_reduce(mean_loss)
            mean_loss /= world
        t_now = time.time()
        tps = (state["tokens"] - tok_log) / max(t_now - t_log, 1e-9)
        t_log, tok_log = t_now, state["tokens"]
        entry = {"step": state["step"], "tokens": state["tokens"], "train_loss": mean_loss.item(),
                 "lr": lr, "tok_per_s": tps, "hours": trained / 3600, "grad_norm": float(grad_norm)}
        state["log"].append(entry)
        log(f"step {state['step']:>6} | tokens {state['tokens'] / 1e6:8.1f}M | loss {entry['train_loss']:.4f} | "
            f"lr {lr:.2e} | {tps:,.0f} tok/s | {trained / 3600:.2f} h")
    if do_eval:
        vl = evaluate()
        state["log"].append({"step": state["step"], "tokens": state["tokens"], "val_loss": vl, "hours": trained / 3600})
        log(f"    validation loss {vl:.4f} (ppl {np.exp(vl):.2f})")
        last_eval = time.time()
    if do_ckpt:
        save_ckpt()
        last_ckpt = time.time()

state["train_seconds"] = base_seconds + (time.time() - loop_start)
vl = evaluate()
state["log"].append({"step": state["step"], "tokens": state["tokens"], "val_loss": vl,
                     "hours": state["train_seconds"] / 3600})
log(f"Finished: step {state['step']}, {state['tokens']:,} tokens this run total, final validation loss {vl:.4f} "
    f"(ppl {np.exp(vl):.2f})")
save_ckpt()
if master:
    os.makedirs(cfg["final_dir"], exist_ok=True)
    torch.save(model.state_dict(), os.path.join(cfg["final_dir"], "pytorch_model.bin"))
    with open(os.path.join(cfg["final_dir"], "config.json"), "w") as f:
        json.dump(cfg["model"], f, indent=2)
    with open(os.path.join(cfg["final_dir"], "training_state.json"), "w") as f:
        json.dump({"step": state["step"], "tokens": state["tokens"], "train_hours": state["train_seconds"] / 3600,
                   "final_val_loss": vl, "annealed": state["train_seconds"] >= total_seconds - 120}, f, indent=2)
if ddp:
    dist.barrier()
    dist.destroy_process_group()

## 10. Continue Training

In [ ]:
if TRAIN_MORE:
    train_hours = min(MAX_TRAIN_HOURS, hours_left() - POST_TRAIN_RESERVE_HOURS)
    print(f"Session time used: {SESSION_LIMIT_HOURS - hours_left():.2f} h -> training for {train_hours:.2f} h")
    assert train_hours >= MIN_TRAIN_HOURS, "Not enough session time left for training."
    train_cfg = dict(
        model=MODEL, train_bin=TRAIN_BIN, val_bin=VAL_BIN, ckpt_path=CKPT_PATH, log_path=TRAIN_LOG_PATH,
        final_dir=CONT_MODEL_DIR, init_from=INIT_FROM, seed=SEED + 1, micro_batch=MICRO_BATCH,
        grad_accum=GRAD_ACCUM, peak_lr=PEAK_LR, min_lr_ratio=MIN_LR_RATIO, warmup_steps=WARMUP_STEPS,
        decay_fraction=DECAY_FRACTION, weight_decay=WEIGHT_DECAY, compile=USE_COMPILE,
        eval_every_sec=1200, ckpt_every_sec=1800, log_every=20, eval_batches=4,
        deadline=time.time() + train_hours * 3600, total_train_seconds=None,
    )
    cfg_path = os.path.join(CODE_DIR, "train_config.json")
    with open(cfg_path, "w") as f:
        json.dump(train_cfg, f, indent=2)
    cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={max(1, N_GPUS)}",
           os.path.join(CODE_DIR, "train.py"), cfg_path]
    env = dict(os.environ, OMP_NUM_THREADS="2", PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in proc.stdout:
        if "barrier(): using the device" in line or "return func(*args" in line:
            continue
        print(line, end="", flush=True)
    if proc.wait() != 0:
        print("Training FAILED - continuing with the first-run model so a submission is still produced.")
        TRAIN_MORE = False
    else:
        print(json.load(open(os.path.join(CONT_MODEL_DIR, "training_state.json"))))
else:
    print("TRAIN_MORE is False -> decoding-only run with the first-run model.")

## 11. Plot Training and Validation Loss

In [ ]:
import matplotlib.pyplot as plt
if TRAIN_MORE and os.path.exists(TRAIN_LOG_PATH):
    tlog = json.load(open(TRAIN_LOG_PATH))
    off = prev_state.get("tokens", 0) / 1e6
    tr = [(off + e["tokens"] / 1e6, e["train_loss"]) for e in tlog if "train_loss" in e]
    va = [(off + e["tokens"] / 1e6, e["val_loss"]) for e in tlog if "val_loss" in e]
    plt.figure(figsize=(10, 5))
    plt.plot(*zip(*tr), label="Training Loss (continued)", alpha=0.8)
    plt.plot(*zip(*va), "o-", label="Validation Loss (continued)")
    if prev_state.get("final_val_loss"):
        plt.axhline(prev_state["final_val_loss"], color="gray", linestyle="--", label="Run 1 final validation loss")
    plt.xlabel("Total training tokens (millions, incl. run 1)")
    plt.ylabel("Loss")
    plt.title("LegalLM Continued Training and Validation Loss")
    plt.legend(); plt.grid(True); plt.tight_layout()
    plt.savefig(LOSS_PLOT_PATH, dpi=200)
    plt.show()
    print("Loss plot saved to:", LOSS_PLOT_PATH)

## 12. Load Models and Compare Validation Loss

In [ ]:
def load_state(model_dir):
    return torch.load(os.path.join(model_dir, "pytorch_model.bin"), map_location="cpu")

val_arr = np.memmap(VAL_BIN, dtype=np.uint16, mode="r")

@torch.no_grad()
def validation_loss(mdl, dev):
    T = MODEL["max_seq_len"]
    starts = np.linspace(0, len(val_arr) - T - 2, 256).astype(np.int64)
    amp = torch.autocast("cuda", dtype=torch.float16) if dev.type == "cuda" else contextlib.nullcontext()
    tot = 0.0
    for s in range(0, len(starts), 16):
        buf = torch.from_numpy(np.stack([val_arr[i:i + T + 1] for i in starts[s:s + 16]]).astype(np.int64)).to(dev)
        with amp:
            tot += mdl(buf[:, :-1], targets=buf[:, 1:]).item() * len(buf)
    return tot / len(starts)

candidates_models = {"run1": load_state(PREV_MODEL_DIR)}
if TRAIN_MORE:
    candidates_models["continued"] = load_state(CONT_MODEL_DIR)
dev0 = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
val_losses = {}
for name, sd in candidates_models.items():
    m = LegalLM(ModelConfig(**MODEL)); m.load_state_dict(sd); m = m.to(dev0).eval()
    val_losses[name] = validation_loss(m, dev0)
    print(f"{name:<10} validation loss {val_losses[name]:.4f} | perplexity {math.exp(val_losses[name]):.2f}")
    del m
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 13. Metrics: BERTScore F1 (primary), ROUGE-L F1, BLEU-4

In [ ]:
from rouge_score import rouge_scorer
import sacrebleu
from bert_score import BERTScorer

bert_scorer = BERTScorer(lang="en", batch_size=64, device=str(dev0))   # roberta-large, layer 17
_rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def compute_metrics(preds, refs):
    rouge_l = sum(_rouge.score(r, p)["rougeL"].fmeasure for p, r in zip(preds, refs)) / len(preds)
    bleu4 = sacrebleu.corpus_bleu(preds, [refs]).score / 100.0
    _, _, f1 = bert_scorer.score(preds, refs)
    return {"bertscore_f1": float(f1.mean()), "rougeL_f1": float(rouge_l), "bleu4": float(bleu4)}

def by_kind_f1(preds, refs, kinds):
    _, _, f1 = bert_scorer.score(preds, refs)
    f1 = f1.numpy()
    return {k: float(f1[[i for i, x in enumerate(kinds) if x == k]].mean()) for k in sorted(set(kinds))}
print("Metric functions ready.")

## 14. Generation (both GPUs) and Fast MBR

#### `generate()` spreads batches over one model copy per GPU. `pairwise_f1_fast()` encodes every MBR candidate once with BERTScore's own model and applies BERTScore's greedy matching to every pair; the next cell checks it against `bert_score` itself and falls back to the slow path if they ever differ.

In [ ]:
import threading, queue
from collections import defaultdict

PROMPT_MAX_TOKENS = MODEL["max_seq_len"] - MAX_NEW_TOKENS


def clip_words(text, n=None):
    return " ".join(text.split()[:n or TARGET_WORDS])


def encode_prompts(texts):
    encs = ENCODE_BATCH([t.rstrip() for t in texts])
    return [(e.ids[-PROMPT_MAX_TOKENS:] if e.ids else [EOS_ID]) for e in encs]


def make_replicas(state_dict, cfg_dict):
    """One fp32 copy of the model per GPU (or one on CPU)."""
    devs = [torch.device(f"cuda:{i}") for i in range(max(1, N_GPUS))] if torch.cuda.is_available() else [torch.device("cpu")]
    reps = []
    for d in devs:
        m = LegalLM(ModelConfig(**cfg_dict))
        m.load_state_dict(state_dict)
        reps.append((m.to(d).eval(), d))
    return reps


@torch.no_grad()
def generate_ids(mdl, dev, prompt_ids, greedy=True, temperature=1.0, top_p=1.0, rep_penalty=1.0,
                 rep_scope="generated", no_repeat_ngram=0, seed=SEED):
    """Batched generation with KV cache on one device. Left padding, per-sequence RoPE positions,
    EOS blocked so every continuation reaches TARGET_WORDS words."""
    amp = torch.autocast("cuda", dtype=torch.float16) if dev.type == "cuda" else contextlib.nullcontext()
    Bsz = len(prompt_ids)
    L = max(len(p) for p in prompt_ids)
    ids = torch.full((Bsz, L), EOS_ID, dtype=torch.long)
    valid = torch.zeros((Bsz, L), dtype=torch.bool)
    for i, p in enumerate(prompt_ids):
        ids[i, L - len(p):] = torch.tensor(p, dtype=torch.long)
        valid[i, L - len(p):] = True
    ids, valid = ids.to(dev), valid.to(dev)
    positions = (valid.long().cumsum(1) - 1).clamp(min=0)
    causal = torch.tril(torch.ones(L, L, dtype=torch.bool, device=dev))
    eye = torch.eye(L, dtype=torch.bool, device=dev)
    mask = ((causal[None] & valid[:, None, :]) | eye[None])[:, None]
    caches = [dict() for _ in mdl.blocks]
    gen = torch.Generator(device=dev).manual_seed(int(seed))
    V = mdl.cfg.vocab_size

    with amp:
        logits = mdl(ids, positions=positions, attn_mask=mask, caches=caches, last_only=True)[:, -1].float()
    next_pos = valid.sum(1)
    key_valid = valid
    presence = torch.zeros(Bsz, V, dtype=torch.bool, device=dev)
    if rep_penalty != 1.0 and rep_scope == "all":
        presence.scatter_(1, ids, True)
    if no_repeat_ngram:
        n = no_repeat_ngram
        hist = [list(p) for p in prompt_ids]
        tables = []
        for h in hist:
            tab = {}
            for j in range(len(h) - n + 1):
                tab.setdefault(tuple(h[j:j + n - 1]), set()).add(h[j + n - 1])
            tables.append(tab)

    out = torch.empty(Bsz, MAX_NEW_TOKENS, dtype=torch.long, device=dev)
    steps = 0
    for t in range(MAX_NEW_TOKENS):
        logits[:, EOS_ID] = float("-inf")
        if rep_penalty != 1.0:
            penalised = torch.where(logits > 0, logits / rep_penalty, logits * rep_penalty)
            logits = torch.where(presence, penalised, logits)
        if no_repeat_ngram:
            for b in range(Bsz):
                if len(hist[b]) >= n - 1:
                    banned = tables[b].get(tuple(hist[b][len(hist[b]) - n + 1:]))
                    if banned:
                        logits[b, list(banned)] = float("-inf")
        if greedy:
            nxt = logits.argmax(-1)
        else:
            probs = torch.softmax(logits / temperature, dim=-1)
            sp, si = torch.sort(probs, descending=True, dim=-1)
            sp = sp.masked_fill((sp.cumsum(-1) - sp) > top_p, 0.0)
            nxt = si.gather(1, torch.multinomial(sp, 1, generator=gen)).squeeze(1)
        out[:, t] = nxt
        steps = t + 1
        if rep_penalty != 1.0:
            presence[torch.arange(Bsz, device=dev), nxt] = True
        if no_repeat_ngram:
            for b, tokid in enumerate(nxt.tolist()):
                h = hist[b]
                h.append(tokid)
                if len(h) >= n:
                    tables[b].setdefault(tuple(h[-n:-1]), set()).add(tokid)
        if steps >= 128 and steps % 16 == 0:
            texts = tok.decode_batch(out[:, :steps].tolist())
            if all(len(x.split()) > TARGET_WORDS for x in texts):
                break
        if steps == MAX_NEW_TOKENS:
            break
        key_valid = torch.cat([key_valid, torch.ones(Bsz, 1, dtype=torch.bool, device=dev)], dim=1)
        with amp:
            logits = mdl(nxt[:, None], positions=next_pos[:, None], attn_mask=key_valid[:, None, None, :],
                         caches=caches, last_only=True)[:, -1].float()
        next_pos = next_pos + 1
    return [clip_words(x) for x in tok.decode_batch(out[:, :steps].tolist())]


def generate(replicas, prompts, config, n=1, seed=SEED, verbose=False):
    """Returns, for every prompt, a list of n continuations. Batches are spread over all GPUs
    (one worker thread per model replica). Deterministic: each batch has its own seed."""
    ids = encode_prompts(prompts)
    order = sorted(range(len(ids)), key=lambda i: len(ids[i]))
    per_batch = max(1, SEQ_BATCH // n)
    batches = [order[s:s + per_batch] for s in range(0, len(order), per_batch)]
    results = [None] * len(ids)
    jobs = queue.Queue()
    for b, idx in enumerate(batches):
        jobs.put((b, idx))
    errors, done = [], [0]
    lock = threading.Lock()
    t0 = time.time()

    def worker(mdl, dev):
        try:
            while True:
                try:
                    b, idx = jobs.get_nowait()
                except queue.Empty:
                    return
                flat = [ids[i] for i in idx for _ in range(n)]
                texts = generate_ids(mdl, dev, flat, seed=seed * 100003 + b, **config)
                with lock:
                    for k, i in enumerate(idx):
                        results[i] = texts[k * n:(k + 1) * n]
                    done[0] += len(idx)
                    if verbose and (b % 10 == 0):
                        eta = (time.time() - t0) / done[0] * (len(ids) - done[0])
                        print(f"  {done[0]}/{len(ids)} prompts ({100 * done[0] / len(ids):.1f}%)  ETA {eta / 60:.1f} min", flush=True)
        except Exception as e:
            errors.append(e)

    threads = [threading.Thread(target=worker, args=r) for r in replicas]
    for th in threads:
        th.start()
    for th in threads:
        th.join()
    if errors:
        raise errors[0]
    return results


def generate_one(replicas, prompts, config, seed=SEED, verbose=False):
    return [r[0] for r in generate(replicas, prompts, config, n=1, seed=seed, verbose=verbose)]


# ---------------------------------------------------------------- fast MBR utility (exact BERTScore F1)
try:
    from bert_score.utils import get_bert_embedding
except Exception:
    get_bert_embedding = None


def _bs_idf_dict():
    t = bert_scorer._tokenizer
    d = defaultdict(lambda: 1.0)
    d[t.sep_token_id] = 0
    d[t.cls_token_id] = 0
    return d


@torch.no_grad()
def pairwise_f1_fast(cand_lists, batch_size=96):
    """BERTScore F1 between every pair of candidates of each prompt, encoding every candidate ONCE
    (bert_score.score would re-encode each one for every pair). Same model, layer, tokenisation
    and greedy matching as bert_score; verified against it below."""
    flat = [c for cands in cand_lists for c in cands]
    emb, mask, idf = get_bert_embedding(flat, bert_scorer._model, bert_scorer._tokenizer, BS_IDF,
                                        batch_size=batch_size, device=bert_scorer.device)
    emb = emb / emb.norm(dim=-1, keepdim=True)
    mask = mask.to(emb.device).float()
    idf = idf.to(emb.device)
    idf = idf / idf.sum(1, keepdim=True)
    out, s = [], 0
    for cands in cand_lists:
        C = len(cands)
        e, m, w = emb[s:s + C], mask[s:s + C], idf[s:s + C]
        s += C
        sim = torch.einsum("ikd,jld->ijkl", e, e)
        sim = sim * (m[:, None, :, None] * m[None, :, None, :])
        P = (sim.max(dim=3).values * w[:, None, :]).sum(-1)      # candidate i as hypothesis
        R = (sim.max(dim=2).values * w[None, :, :]).sum(-1)      # candidate j as reference
        F = torch.nan_to_num(2 * P * R / (P + R))
        out.append(F.float().cpu())
    return out


def pairwise_f1_slow(cand_lists):
    out = []
    for cands in cand_lists:
        C = len(cands)
        a = [cands[i] for i in range(C) for j in range(C) if i != j]
        b = [cands[j] for i in range(C) for j in range(C) if i != j]
        _, _, f = bert_scorer.score(a, b)
        F = torch.zeros(C, C)
        k = 0
        for i in range(C):
            for j in range(C):
                if i != j:
                    F[i, j] = f[k]
                    k += 1
        out.append(F)
    return out


def pairwise_f1(cand_lists, chunk=8):
    fn = pairwise_f1_fast if FAST_MBR else pairwise_f1_slow
    res = []
    for s in range(0, len(cand_lists), chunk):
        res.extend(fn(cand_lists[s:s + chunk]))
    return res


def mbr_pick(cands, F, n_bases=0):
    """Candidate with the highest mean BERTScore F1 against the SAMPLED candidates (the first n_bases
    entries are deterministic greedy outputs: they can be chosen but do not count as evidence, so two
    identical greedy outputs cannot vote for each other). Short candidates are used only as a last resort."""
    C = len(cands)
    ev = torch.zeros(C, C)
    ev[:, n_bases:] = 1.0
    ev.fill_diagonal_(0.0)
    util = (F * ev).sum(1) / ev.sum(1).clamp(min=1)
    ok = torch.tensor([len(c.split()) >= TARGET_WORDS for c in cands])
    if ok.any():
        util = torch.where(ok, util, torch.full_like(util, -1e9))
    return int(util.argmax())


def mbr_select(cand_lists, n_bases=2):
    Fs = pairwise_f1(cand_lists)
    return [c[mbr_pick(c, F, n_bases)] for c, F in zip(cand_lists, Fs)]

In [ ]:
BS_IDF = _bs_idf_dict() if get_bert_embedding is not None else None
FAST_MBR = False
if get_bert_embedding is not None:
    try:
        probe = [[eval_refs[i], eval_refs[i + 1][:400], eval_prompts[i][-700:], eval_refs[i + 2]] for i in range(3)]
        FAST_MBR = True
        fast = pairwise_f1(probe)
        FAST_MBR = False
        slow = pairwise_f1(probe)
        diff = max(float(((a - b) * (1 - torch.eye(len(a)))).abs().max()) for a, b in zip(fast, slow))
        FAST_MBR = diff < 1e-3
        print(f"Fast MBR vs bert_score: max difference {diff:.2e} -> {'using fast path' if FAST_MBR else 'using slow path'}")
    except Exception as e:
        FAST_MBR = False
        print("Fast MBR unavailable, using bert_score directly:", repr(e)[:200])
if not FAST_MBR:   # pairwise scoring through bert_score is slow -> keep candidate sets small
    MBR_POOL_SAMPLES = min(MBR_POOL_SAMPLES, 8)
    MBR_MAX_SAMPLES = min(MBR_MAX_SAMPLES, 8)
    print("Slow MBR path: candidate counts reduced to", MBR_POOL_SAMPLES)

## 15. Decoding Search on Judgment Prompts

#### (a) greedy + repetition penalty for each model -> pick the model. (b) one pool of sampled candidates per prompt; MBR variants (temperature, number of samples) are scored from subsets of that pool, so nothing is generated twice.

In [ ]:
sweep_prompts, sweep_refs = eval_prompts[:SWEEP_N], eval_refs[:SWEEP_N]
sweep = {}
plain_outputs = {}
for mname, sd in candidates_models.items():
    reps = make_replicas(sd, MODEL)
    for rp in REP_PENALTIES:
        name = f"{mname} | greedy_rep{rp}"
        t0 = time.time()
        outs = generate_one(reps, sweep_prompts, dict(greedy=True, rep_penalty=rp))
        secs = time.time() - t0
        m = compute_metrics(outs, sweep_refs)
        m["sec_per_seq"] = secs / len(outs)
        sweep[name], plain_outputs[name] = m, outs
        print(f"{name:<34} BERTScore F1={m['bertscore_f1']:.4f}  ROUGE-L={m['rougeL_f1']:.4f}  "
              f"BLEU-4={m['bleu4']:.4f}  ({secs:.0f}s)", flush=True)
    del reps
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

best_plain = max(plain_outputs, key=lambda k: sweep[k]["bertscore_f1"])
BEST_MODEL = best_plain.split(" | ")[0]
BEST_REP = float(best_plain.split("greedy_rep")[1])
ranked = sorted([k for k in plain_outputs if k.startswith(BEST_MODEL)], key=lambda k: -sweep[k]["bertscore_f1"])
SECOND_REP = float(ranked[1].split("greedy_rep")[1]) if len(ranked) > 1 else BEST_REP
print(f"\nSelected model: {BEST_MODEL} | best plain decoding: greedy, repetition penalty {BEST_REP} "
      f"(second best {SECOND_REP})")
replicas = make_replicas(candidates_models[BEST_MODEL], MODEL)

In [ ]:
# One candidate pool per sweep prompt: 2 greedy bases + MBR_POOL_SAMPLES samples at each temperature
base_a = plain_outputs[f"{BEST_MODEL} | greedy_rep{BEST_REP}"]
base_b = plain_outputs[f"{BEST_MODEL} | greedy_rep{SECOND_REP}"]
pool = {}
t0 = time.time()
for temp in MBR_POOL_TEMPS:
    pool[temp] = generate(replicas, sweep_prompts,
                          dict(greedy=False, temperature=temp, top_p=0.95, rep_penalty=BEST_REP),
                          n=MBR_POOL_SAMPLES, seed=SEED + int(temp * 1000))
gen_secs = time.time() - t0
SEC_PER_SEQ = gen_secs / (len(sweep_prompts) * MBR_POOL_SAMPLES * len(MBR_POOL_TEMPS))
print(f"Pool generated in {gen_secs / 60:.1f} min -> {SEC_PER_SEQ * 1000:.0f} ms per sequence (all GPUs)")

def pool_variant(temps, n):
    cands = []
    for i in range(len(sweep_prompts)):
        c = [base_a[i], base_b[i]]
        for tmp in temps:
            c += pool[tmp][i][:n]
        cands.append(c)
    return cands

variants = {}
for n in sorted({min(8, MBR_POOL_SAMPLES), min(16, MBR_POOL_SAMPLES), MBR_POOL_SAMPLES}):
    for temp in MBR_POOL_TEMPS:
        variants[f"mbr | t{temp} x{n}"] = ([temp], n)
    if n * len(MBR_POOL_TEMPS) <= MBR_MAX_SAMPLES:
        variants[f"mbr | mixed x{n * len(MBR_POOL_TEMPS)}"] = (MBR_POOL_TEMPS, n)

t0 = time.time()
mbr_preds = {}
for name, (temps, n) in variants.items():
    preds = mbr_select(pool_variant(temps, n))
    m = compute_metrics(preds, sweep_refs)
    sweep[name], mbr_preds[name] = m, preds
    print(f"{name:<34} BERTScore F1={m['bertscore_f1']:.4f}  ROUGE-L={m['rougeL_f1']:.4f}  BLEU-4={m['bleu4']:.4f}", flush=True)
print(f"MBR scoring: {(time.time() - t0) / 60:.1f} min ({'fast' if FAST_MBR else 'slow'} path)")

best_mbr = max(mbr_preds, key=lambda k: sweep[k]["bertscore_f1"])
MBR_TEMPS, MBR_N = variants[best_mbr]
print(f"\nBest plain : {best_plain}  {sweep[best_plain]['bertscore_f1']:.4f}")
print(f"Best MBR   : {best_mbr}  {sweep[best_mbr]['bertscore_f1']:.4f}")

## 16. Held-Out Report (all evaluation pairs, per court type)

#### Prompts beyond the sweep subset were not used to make any choice, so the "unseen" rows are an unbiased estimate.

In [ ]:
rest = list(range(SWEEP_N, len(eval_pairs)))
report = {}
if rest:
    rest_prompts = [eval_prompts[i] for i in rest]
    rest_refs = [eval_refs[i] for i in rest]
    rest_kinds = [eval_kinds[i] for i in rest]
    plain_rest = generate_one(replicas, rest_prompts, dict(greedy=True, rep_penalty=BEST_REP))
    report["plain (unseen prompts)"] = compute_metrics(plain_rest, rest_refs)
    cands = [[] for _ in rest_prompts]
    for temp in MBR_TEMPS:
        more = generate(replicas, rest_prompts, dict(greedy=False, temperature=temp, top_p=0.95,
                        rep_penalty=BEST_REP), n=MBR_N, seed=SEED + 99 + int(temp * 1000))
        cands = [c + d for c, d in zip(cands, more)]
    plain_rest_b = generate_one(replicas, rest_prompts, dict(greedy=True, rep_penalty=SECOND_REP))
    mbr_rest = mbr_select([[a, b] + c for a, b, c in zip(plain_rest, plain_rest_b, cands)])
    report["MBR (unseen prompts)"] = compute_metrics(mbr_rest, rest_refs)
    report["MBR per court (unseen)"] = by_kind_f1(mbr_rest, rest_refs, rest_kinds)
report["plain (sweep prompts)"] = sweep[best_plain]
report["MBR (sweep prompts)"] = sweep[best_mbr]
print("=" * 78)
for k, v in report.items():
    if "bertscore_f1" in v:
        print(f"{k:<32} BERTScore F1={v['bertscore_f1']:.4f}  ROUGE-L={v['rougeL_f1']:.4f}  BLEU-4={v['bleu4']:.4f}")
    else:
        print(f"{k:<32} " + "  ".join(f"{kk}={vv:.4f}" for kk, vv in v.items()))
print("=" * 78)
print("Run 1 reference: leaderboard 0.8427; run-1 local Supreme Court subset 0.8450")

## 17. Test Set: Fallback Submission (plain decoding), then MBR Submission

#### The plain-decoding `submission.zip` is written first, so a valid file exists even if the session runs out of time. The number of MBR samples is then chosen to fit the time left.

In [ ]:
test_items = list(test_data.items())
test_prompts = [r["text"] for _, r in test_items]

def save_submission(outs, tag):
    preds = {str(eid): {"text": o} for (eid, _), o in zip(test_items, outs)}
    with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
        json.dump(preds, f, ensure_ascii=False, indent=2)
    with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(PREDICTION_PATH, arcname="prediction.json")
    shutil.copy(SUBMISSION_ZIP, os.path.join(OUTPUT_DIR, f"submission_{tag}.zip"))
    short = sum(len(o.split()) < TARGET_WORDS for o in outs)
    print(f"Saved submission ({tag}): {len(preds)} predictions, {short} shorter than {TARGET_WORDS} words")

t0 = time.time()
test_base_a = generate_one(replicas, test_prompts, dict(greedy=True, rep_penalty=BEST_REP), verbose=True)
save_submission(test_base_a, "plain")
test_base_b = generate_one(replicas, test_prompts, dict(greedy=True, rep_penalty=SECOND_REP))
print(f"Plain decoding of the test set: {(time.time() - t0) / 60:.1f} min | session time left {hours_left():.2f} h")

# Export the selected model with tokenizer and code
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)
torch.save(candidates_models[BEST_MODEL], os.path.join(FINAL_MODEL_DIR, "pytorch_model.bin"))
with open(os.path.join(FINAL_MODEL_DIR, "config.json"), "w") as f:
    json.dump(MODEL, f, indent=2)
shutil.copy(TOKENIZER_PATH, os.path.join(FINAL_MODEL_DIR, "tokenizer.json"))
for fn in ["model.py", "train.py"]:
    shutil.copy(os.path.join(CODE_DIR, fn), os.path.join(FINAL_MODEL_DIR, fn))

In [ ]:
use_mbr = sweep[best_mbr]["bertscore_f1"] > sweep[best_plain]["bertscore_f1"]
test_mbr_n = 0
if use_mbr:
    # time per test prompt for n samples ~ gen time + scoring time (scoring measured as ~15% of generation)
    budget_s = (hours_left() - FINAL_MARGIN_HOURS) * 3600
    per_sample_s = SEC_PER_SEQ * (1.15 if FAST_MBR else 3.0) * len(test_prompts)
    fit = int(budget_s / per_sample_s)
    want = MBR_N * len(MBR_TEMPS)
    test_mbr_n = min(MBR_MAX_SAMPLES, max(want, (fit // len(MBR_TEMPS)) * len(MBR_TEMPS)), fit)
    print(f"MBR samples chosen on held-out: {want} | fit in time left: {fit} -> using {test_mbr_n}")
    if test_mbr_n < MBR_MIN_SAMPLES:
        print("Not enough time for MBR - keeping the plain-decoding submission.")
        use_mbr = False

if use_mbr:
    per_temp = test_mbr_n // len(MBR_TEMPS)
    test_cands = [[a, b] for a, b in zip(test_base_a, test_base_b)]
    t0 = time.time()
    for temp in MBR_TEMPS:
        print(f"Sampling {per_temp} candidates per prompt at temperature {temp}")
        samples = generate(replicas, test_prompts, dict(greedy=False, temperature=temp, top_p=0.95,
                           rep_penalty=BEST_REP), n=per_temp, seed=SEED + 5000 + int(temp * 1000), verbose=True)
        for c, s in zip(test_cands, samples):
            c.extend(s)
    print(f"Candidates generated in {(time.time() - t0) / 60:.1f} min | time left {hours_left():.2f} h")
    with open(CANDIDATES_PATH, "w", encoding="utf-8") as f:
        json.dump({str(eid): c for (eid, _), c in zip(test_items, test_cands)}, f, ensure_ascii=False)
    t0 = time.time()
    test_final = mbr_select(test_cands)
    print(f"MBR selection: {(time.time() - t0) / 60:.1f} min")
    save_submission(test_final, "mbr")
    final_decoding = f"MBR: 2 greedy (rep {BEST_REP}, {SECOND_REP}) + {test_mbr_n} samples at {MBR_TEMPS}"
else:
    final_decoding = f"greedy, repetition penalty {BEST_REP}"

with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "parameters": model_size, "architecture": MODEL, "selected_model": BEST_MODEL,
        "validation_loss": val_losses, "final_decoding": final_decoding,
        "report": report, "decoding_sweep": sweep, "num_eval_pairs": len(eval_pairs),
        "continued_training": json.load(open(os.path.join(CONT_MODEL_DIR, "training_state.json"))) if TRAIN_MORE else None,
    }, f, indent=2)
print("Final decoding:", final_decoding)
print(f"Session time used: {SESSION_LIMIT_HOURS - hours_left():.2f} h")
print("Upload this file to Codabench:", SUBMISSION_ZIP)